In [1]:
import pandas as pd
import re
#读取原始数据
df = pd.read_csv(
    "ADHD_OpenAlex数据集.csv"
)
print("原始数据量：", len(df))
#一、删除重复数据
#1. DOI 去重
# DOI 是论文唯一标识
df = df.drop_duplicates(
    subset="doi"
)
#2. 标题去重
# 防止重复索引文献
df = df.drop_duplicates(
    subset="title"
)
print("去重后数据量：", len(df))
#二、缺失值处理
#删除标题为空
df = df.dropna(
    subset=["title"]
)
# 删除全空格标题
df = df[
    df["title"].astype(str).str.strip() != ""
]
#删除摘要为空
# 后续文本分析必须使用摘要
df = df.dropna(
    subset=["abstract"]
)
# 删除全空格摘要
df = df[
    df["abstract"].astype(str).str.strip() != ""
]
print("缺失值处理后数据量：", len(df))
#三、异常文本处理
#删除异常短标题
# 避免无意义记录
df = df[
    df["title"].astype(str).str.len() >= 5
]
#删除异常摘要
df = df[
    df["abstract"].astype(str).str.len() >= 30
]
print("异常文本处理后数据量：", len(df))
#四、统一文本格式
def clean_text(text):
    # 空值处理
    if pd.isna(text):
        return ""
    # 转字符串
    text = str(text)
    # 转小写
    text = text.lower()
    # 删除网址
    text = re.sub(
        r"http\S+",
        "",
        text
    )
    # 删除换行
    text = re.sub(
        r"\n",
        " ",
        text
    )
    # 删除制表符
    text = re.sub(
        r"\t",
        " ",
        text
    )
    # 删除多余空格
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    return text.strip()
#清洗标题
df["title_clean"] = df[
    "title"
].apply(clean_text)
#清洗摘要
df["abstract_clean"] = df[
    "abstract"
].apply(clean_text)
#五、统一字段类型
numeric_cols = [
    "publication_year",
    "cited_by_count",
    "authors_count",
    "institutions_count",
    "countries_count",
    "concepts_count",
    "topics_count",
    "title_length",
    "abstract_length",
    "referenced_works_count",
    "recent_citations",
    "citation_percentile"
]
# 数值类型转换
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )
# 布尔类型统一
bool_cols = [
    "is_oa",
    "international_collab",
    "multi_institution",
    "is_highly_cited"
]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str)
#六、简单数据质量检查
print("\n各字段缺失值情况：")
print(
    df.isnull().sum()
)
print("\n数据类型：")
print(
    df.dtypes
)
print("\n年份分布：")
print(
    df["publication_year"]
    .value_counts()
)
#七、保存清洗结果
df.to_csv(
    "ADHD_清洗后数据.csv",
    index=False,
    encoding="utf-8-sig"
)
#八、输出结果
print("\n数据清洗完成！")
print(
    "最终数据量：",
    len(df)
)
print(
    "\n文件已保存：ADHD_清洗后数据.csv"
)

C:\Users\21116\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


原始数据量： 8338
去重后数据量： 8192
缺失值处理后数据量： 6559
异常文本处理后数据量： 6551

各字段缺失值情况：
paper_id                     0
title                        0
abstract                     0
publication_year             0
doi                          1
journal                    306
cited_by_count               0
citation_percentile       3072
recent_citations             0
is_highly_cited              0
is_oa                        0
authors                     58
authors_count                0
institutions               942
institutions_count           0
countries                  947
countries_count              0
international_collab         0
multi_institution            0
concepts                     0
concepts_count               0
topics                       0
topics_count                 0
title_length                 0
abstract_length              0
referenced_works_count       0
title_clean                  0
abstract_clean               0
dtype: int64

数据类型：
paper_id                      str
title    